# Dana’s Bakery — deterministic LangGraph order system

Implements the specification in prompt.md with LangGraph StateGraph and no LLM calls. Stock is an in-memory snapshot.


In [ ]:
%pip install -q langgraph

from typing import TypedDict, List
from datetime import date
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END


## State and configuration


In [ ]:
class OrderState(TypedDict, total=False):
    item: str
    quantity: int
    day: str
    city: str
    status: str
    delivery: float
    total: float
    message: str
    steps: List[str]
    current_item: str
    attempt_count: int
    _delivery_note: str
    _pricing_note: str

MENU = {
    "challah": {"price": 18, "stock": 10},
    "sourdough": {"price": 25, "stock": 4},
    "rugelach": {"price": 30, "stock": 3},
    "babka": {"price": 45, "stock": 0},
    "croissants": {"price": None, "stock": 0},
}
SUBSTITUTION_CHAIN = ["babka", "rugelach", "sourdough", "challah"]
MAX_SUBSTITUTION_ATTEMPTS = 2

def normalized(value):
    return value.strip().casefold() if isinstance(value, str) else ""

def is_friday(day_value):
    value = normalized(day_value)
    if value in {"friday", "fri"}:
        return True
    try:
        return date.fromisoformat(value).weekday() == 4
    except (TypeError, ValueError):
        return False

def visit(state, node, **updates):
    """Append a diary entry every time a graph node is entered."""
    return {**updates, "steps": [*state.get("steps", []), node]}


## Nodes and conditional routing


In [ ]:
def validate_order(state):
    requested = normalized(state.get("item", ""))
    quantity = state.get("quantity")
    valid_quantity = isinstance(quantity, int) and not isinstance(quantity, bool) and quantity > 0
    if requested not in MENU:
        return visit(state, "validate_order", item=requested, current_item=requested, status="invalid", message=f"Unknown menu item: {requested or '(empty)' }.")
    if not valid_quantity:
        return visit(state, "validate_order", item=requested, current_item=requested, status="invalid", message="Quantity must be a positive integer.")
    if requested == "croissants":
        return visit(state, "validate_order", item=requested, current_item=requested, status="invalid", message="Croissants have been discontinued and are unavailable.")
    return visit(state, "validate_order", item=requested, current_item=requested, status="pending", attempt_count=0, delivery=0.0, total=0.0, message="")

def check_stock(state):
    item = state.get("current_item", "")
    available = item in MENU and MENU[item]["price"] is not None and MENU[item]["stock"] >= state["quantity"]
    return visit(state, "check_stock", status="available" if available else "out_of_stock")

def handle_substitution(state):
    current, attempts = state.get("current_item", ""), state.get("attempt_count", 0)
    if current not in SUBSTITUTION_CHAIN or attempts >= MAX_SUBSTITUTION_ATTEMPTS:
        return visit(state, "handle_substitution", status="unfulfillable")
    next_index = SUBSTITUTION_CHAIN.index(current) + 1
    if next_index >= len(SUBSTITUTION_CHAIN):
        return visit(state, "handle_substitution", status="unfulfillable")
    return visit(state, "handle_substitution", current_item=SUBSTITUTION_CHAIN[next_index], attempt_count=attempts + 1, status="pending")

def calculate_pricing(state):
    subtotal = MENU[state["current_item"]]["price"] * state["quantity"]
    city = normalized(state.get("city", ""))
    if not city:
        delivery, delivery_note = 0.0, "Pickup selected; no delivery fee."
    elif city == "jerusalem":
        delivery, delivery_note = 20.0, "Delivery to Jerusalem: 20 NIS."
    else:
        delivery = 0.0
        delivery_note = f"Delivery outside Jerusalem ({state.get('city', '').strip()}) was converted to pickup; no delivery fee."
    friday = is_friday(state.get("day", ""))
    discounted = subtotal > 150 and not friday
    goods_total = round(subtotal * 0.85, 2) if discounted else float(subtotal)
    total = round(goods_total + delivery, 2)
    pricing_note = f"Avi’s 15% discount applied to baked goods ({subtotal} NIS)." if discounted else ("No discount: Friday orders are excluded." if subtotal > 150 and friday else "No discount: baked-goods subtotal is not over 150 NIS.")
    return visit(state, "calculate_pricing", status="priced", delivery=delivery, total=total, _delivery_note=delivery_note, _pricing_note=pricing_note)

def finalize_response(state):
    original, fulfilled = state["item"], state["current_item"]
    substitute = f" Requested {original}; substituted with {fulfilled} to fulfill the full quantity." if fulfilled != original else ""
    message = f"Order accepted: {state['quantity']} x {fulfilled}. {state['_delivery_note']} {state['_pricing_note']}{substitute} Total: {state['total']:.2f} NIS."
    return visit(state, "finalize_response", status="accepted", item=fulfilled, message=message)

def reject_order(state):
    if state.get("status") == "invalid":
        status, message = "rejected", state.get("message", "Order input is invalid.")
    else:
        status = "cancelled"
        message = f"Order cancelled: unable to fulfill the full quantity of {state.get('quantity')} {state.get('item')} after {state.get('attempt_count', 0)} substitution attempt(s)."
    return visit(state, "reject_order", status=status, message=message, delivery=0.0, total=0.0)

def after_validation(state):
    return "check_stock" if state["status"] == "pending" else "reject_order"
def after_stock(state):
    return "calculate_pricing" if state["status"] == "available" else "handle_substitution"
def after_substitution(state):
    return "check_stock" if state["status"] == "pending" else "reject_order"

builder = StateGraph(OrderState)
for name, fn in [("validate_order", validate_order), ("check_stock", check_stock), ("handle_substitution", handle_substitution), ("calculate_pricing", calculate_pricing), ("finalize_response", finalize_response), ("reject_order", reject_order)]:
    builder.add_node(name, fn)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", after_validation, {"check_stock":"check_stock", "reject_order":"reject_order"})
builder.add_conditional_edges("check_stock", after_stock, {"calculate_pricing":"calculate_pricing", "handle_substitution":"handle_substitution"})
builder.add_conditional_edges("handle_substitution", after_substitution, {"check_stock":"check_stock", "reject_order":"reject_order"})
builder.add_edge("calculate_pricing", "finalize_response")
builder.add_edge("finalize_response", END)
builder.add_edge("reject_order", END)
graph = builder.compile()

def place_order(item, quantity, day, city=""):
    initial = {"item":item, "quantity":quantity, "day":day, "city":city, "steps":[]}
    result = graph.invoke(initial)
    return {key:result[key] for key in ("status", "item", "delivery", "total", "message", "steps")}


## Graph visualization


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


## Execution path examples


In [ ]:
from pprint import pprint

cases = {
    "1. Standard Jerusalem delivery": ("challah", 2, "Monday", "Jerusalem"),
    "2. Pickup (empty city)": ("sourdough", 1, "Tuesday", ""),
    "3. Outside Jerusalem converts to pickup": ("rugelach", 2, "Wednesday", "Tel Aviv"),
    "4. Babka substitutes successfully": ("babka", 2, "Monday", "Jerusalem"),
    "5. Cancel after two failed substitution attempts": ("babka", 5, "Monday", ""),
    "6a. Invalid negative quantity": ("challah", -1, "Monday", "Jerusalem"),
    "6b. Unknown menu item": ("danish", 1, "Monday", ""),
    "7a. Large weekday order gets discount": ("challah", 10, "Thursday", "Jerusalem"),
    "7b. Large Friday order is excluded": ("challah", 10, "Friday", "Jerusalem"),
}
results = {name: place_order(*args) for name, args in cases.items()}
for name, result in results.items():
    print("\n" + name)
    pprint(result)


## Assertions for specified behavior


In [ ]:
assert results["1. Standard Jerusalem delivery"]["status"] == "accepted"
assert results["1. Standard Jerusalem delivery"]["delivery"] == 20.0
assert results["1. Standard Jerusalem delivery"]["total"] == 56.0
assert results["2. Pickup (empty city)"]["delivery"] == 0.0
assert results["3. Outside Jerusalem converts to pickup"]["delivery"] == 0.0
assert "converted to pickup" in results["3. Outside Jerusalem converts to pickup"]["message"]
assert "Rugelach" in results["4. Babka substitutes successfully"]["message"]
assert results["4. Babka substitutes successfully"]["total"] == 80.0
assert results["5. Cancel after two failed substitution attempts"]["status"] == "cancelled"
assert results["5. Cancel after two failed substitution attempts"]["steps"].count("handle_substitution") == 3
assert results["6a. Invalid negative quantity"]["status"] == "rejected"
assert results["6b. Unknown menu item"]["status"] == "rejected"
assert results["7a. Large weekday order gets discount"]["total"] == 173.0
assert results["7b. Large Friday order is excluded"]["total"] == 200.0
assert all(r["steps"][0] == "validate_order" for r in results.values())
assert all(r["steps"][-1] in {"finalize_response", "reject_order"} for r in results.values())
print(f"All {len(results)} scenario checks passed.")


## Production boundary

Production order acceptance should reserve stock atomically with durable order creation. This notebook makes no payment, delivery-provider, or LLM calls.
